# Composite DIMSIM geometry analysis

Imports the exact checked canonical implementation from `CODE_ROOT`.


In [ ]:
BUILD_METADATA = {'schema_version': 1, 'baseline_git_sha': '1f77df1526f79a9b67fd092adad79dbbf24b8d79', 'source_sha256': {'pinyin.py': '6d080de3613cd219edd5553512e17e4fe596a7bb881d59c0f6acd654bb85f1d3', 'objectives.py': '2cdb95c3044103e6138d28038d696e43b200ea7b0e94c3131cac9a3fd3c0df47', 'artifacts.py': '9447013a45a347b7df6735cde9dc7c40f0931d53792165999c482c384321d64d', 'data_contract.py': '5c526086867ed477286c1f369ca78e1ccd44fa52ed8453846d87a664f2971fa3', 'ids.py': '832b20b5f97f8328150a6efd57847d9cc56378b703117da1bde65759f29c37ab', 'experiment.py': '518c606af5b1535d928e3bb3a7425a15db77525ea4bc944f2af74ffa0248ddb0', 'training_data.py': '6b16cc0d66276f782c9e839ec37fe6dbce703723f079ef706ef1be94a74cb684', 'reproducibility.py': 'ef59e1eb6453cfb149d11b6a4f2c7227d1aa8a8b7f354fb1a9a9b6e1590296aa', 'model.py': '34932db3ea2ac8b2bf3e0d2631e563dcbe22d686c3408a0d113cd499ff3630a1', 'data.py': '2be23fc5b5a5916c66074dc447143008719ee3c4b1eb2192b73d5f8b30be5989', 'metrics.py': '10d41722970c5472cfe9c3f09346ae977d95dab520fc8d72a4f26d3e7f935fbc', 'training.py': 'e567ed0eef9646479ce3cf50089ea024b1d12946d18e24444cde5dee00e1108d', 'evaluation/dimsim.py': '38712d92358e8378539935fd669f5e9a375193010007a3798faaea02c88e0195', 'evaluation/sampling.py': '39c473e643f1f30c62951d28d4e5b6004ed819256676ff351355d098ebeab0c8', 'evaluation/bundle.py': 'b0495dcba8fae1f13d79200f8e56faa4607fb84758cf899af1bed3c290bf876e', 'evaluation/representations.py': '8d41e607938a592559f77a4e3900ddda3c15975017d0589fbad59a634c43ddbb', 'evaluation/encoders.py': '85e1342b5a4028f850df26dd48683fb41ba569c38b13a786c5e74fa467f6e0de', 'evaluation/analysis.py': 'c06233981812e189e8b696c47fbe5d53d665a3d20bb611f0db15dffc12345577', 'evaluation/runner.py': 'a48b4e0478cf021a89f460641e94c559ca5f50f223e3dc7596a3a47e53d81e35'}, 'embedded_sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'generator_sha256': 'e62a8ed40d30b04792f6da7e9cdfa7e1d047300dbd4bb8e9c1cb559194f7988e'}


In [ ]:
import importlib.metadata
import subprocess
import sys

INSTALL_DEPENDENCIES = False
DEPENDENCIES = ["dimsim==0.2.2", "pypinyin==0.55.0", "numpy", "scipy", "pandas", "matplotlib"]
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *DEPENDENCIES])

for package in ("dimsim", "pypinyin", "numpy", "scipy", "pandas", "matplotlib"):
    print(package, importlib.metadata.version(package))
if importlib.metadata.version("dimsim") != "0.2.2" or importlib.metadata.version("pypinyin") != "0.55.0":
    raise RuntimeError("Pinned DIMSIM preparation dependencies do not match")


In [ ]:
import hashlib
import json
import sys
from pathlib import Path

def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    current = current.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "phonetics").is_dir():
            return candidate
        child = candidate / "project"
        if (child / "src" / "phonetics").is_dir():
            return child
    raise FileNotFoundError("Could not find project root from current path")

CODE_ROOT = find_project_root()
if any(name == "phonetics" or name.startswith("phonetics.") for name in sys.modules):
    raise RuntimeError("Composite evaluation requires a fresh kernel before importing phonetics")
sys.path.insert(0, str(CODE_ROOT / "src"))
from phonetics.evaluation.runner import run_evaluation
from phonetics.evaluation.analysis import compare_paired_runs
from phonetics.objectives import combined_model_id

for relative, expected in BUILD_METADATA["source_sha256"].items():
    actual = hashlib.sha256((CODE_ROOT / "src" / "phonetics" / relative).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f"Local source differs from notebook build: {relative}")


In [ ]:
import hashlib
from pathlib import Path

EXPERIMENT_ID = "duyu-chinese-lessons-zh-cn-v1"
EXPERIMENT_MANIFEST_SHA256 = "719fc321a121c06a25c5247533564c15ce02f26e6bc82aa20c33fc567a227185"
SEED = 0
COMPONENTS_RUN_ID = "REPLACE_WITH_COMPONENTS_RUN_ID"
COMBINED_RUN_ID = "REPLACE_WITH_COMBINED_RUN_ID"
LAMBDA_COSINE = 1.0

PROJECT_ROOT = find_project_root()
BUNDLE_PATH = PROJECT_ROOT / "datasets" / "evaluations" / "1.0.0"
OUTPUT_ROOT = PROJECT_ROOT / "datasets" / "evaluation-runs"
EXPERIMENT_PATH = PROJECT_ROOT / "datasets" / "experiments" / EXPERIMENT_ID
RUNS_ROOT = PROJECT_ROOT / "runs" / EXPERIMENT_ID

if hashlib.sha256((EXPERIMENT_PATH / "manifest.json").read_bytes()).hexdigest() != EXPERIMENT_MANIFEST_SHA256:
    raise ValueError("Experiment does not match the confirmed training input")
RUNS = [
    {"run_id": "components", "adapter": "bilstm", "experiment_path": EXPERIMENT_PATH, "checkpoint_path": RUNS_ROOT / "components" / COMPONENTS_RUN_ID / f"seed-{SEED}" / "best.pt"},
    {"run_id": combined_model_id(LAMBDA_COSINE), "adapter": "bilstm", "experiment_path": EXPERIMENT_PATH, "checkpoint_path": RUNS_ROOT / combined_model_id(LAMBDA_COSINE) / COMBINED_RUN_ID / f"seed-{SEED}" / "best.pt"},
    {"run_id": "random-seed-42", "adapter": "random", "random_seed": 42, "random_dimension": 1024},
]
DEVICE = "cpu"
BATCH_SIZE = 32
PAIRED_COMPARISONS = [("components", combined_model_id(LAMBDA_COSINE))]


In [ ]:
import torch
print("torch", torch.__version__, "cuda_available", torch.cuda.is_available())
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(f"Selected device {DEVICE!r} requires CUDA, but CUDA is unavailable")


In [ ]:
missing_checkpoints = [str(specification["checkpoint_path"]) for specification in RUNS
                       if specification["adapter"] != "random" and not Path(specification["checkpoint_path"]).is_file()]
if missing_checkpoints:
    raise FileNotFoundError("Training checkpoints are required before evaluation: " + ", ".join(missing_checkpoints))

completed = {}
for specification in RUNS:
    arguments = dict(specification)
    run_id = arguments.pop("run_id")
    adapter = arguments.pop("adapter")
    completed[run_id] = run_evaluation(
        BUNDLE_PATH, output_root=OUTPUT_ROOT, run_id=run_id, adapter=adapter,
        device=DEVICE, batch_size=BATCH_SIZE, code_provenance=BUILD_METADATA["source_sha256"], **arguments,
    )

for left, right in PAIRED_COMPARISONS:
    compare_paired_runs(
        completed[left] / "analysis", completed[right] / "analysis",
        output_dir=OUTPUT_ROOT / f"compare-{left}-vs-{right}",
    )

for run_id, path in completed.items():
    metrics = json.loads((path / "analysis" / "metrics.json").read_text(encoding="utf-8"))
    print(run_id, {level: values["overall"] for level, values in metrics.items()})
